<img src="images/network_prof.png" width="150" alt="Prof looking at a electronic brain" style="float: left; margin-right: 15px; margin-bottom: 10px;"> In this section, we will create real agents to answer students' questions about the network course content.

With the philosophy professor, we saw the basic queries of an LLM, with low-level calls.
This required quite a bit of code, and especially the queries were sequential. When we asked several LLMs to work on the assignment, we had to wait for the response
from the first one to launch the second, when they could have done this work at the same time.

Here is the list of modules we will need in this section

In [ ]:
from dotenv import load_dotenv
import os
from pypdf import PdfReader
from IPython.display import Markdown, display 


from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput, set_tracing_export_api_key, ModelSettings

import gradio
import asyncio
import requests 


## Retrieve the content of the book "Programmer l'Internet des Objets"

First, we will convert the first pages of the book to text.

In [ ]:
book = PdfReader("./PLIDO_BOOK_en.pdf")
book_content = ""
for page in book.pages:
    text = page.extract_text()
    book_content += text

## Give the information to the LLM

<img src="images/agent.png" width="150" alt="One agent" style="float: left; margin-right: 15px; margin-bottom: 10px;">Once the API keys are loaded for several servers (we will use the University of Rennes one by default, but using Gemini is also possible). We provide the URI and Token for the service, then in a second step, we specify the LLM model used. If we used OpenAI by default, these lines would be unnecessary. When calling ```Agent```, in the case of OpenAI, the model name is directly indicated in a string.

When creating the Agent, we give it an identifier for traces, then the instructions that will indicate its role, the limits of responses and the actions it will have to take in certain cases. These instructions also contain the entirety of the book in ASCII. Note that we ask the LLM not to try to answer in detail if the answer is not found in the book.

In [ ]:
load_dotenv(override=True)

rennes_api_key = os.getenv("RENNES_API_KEY")
if not rennes_api_key:
    print("RENNES_API_KEY is missing")
    exit(1)

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=rennes_api_key)
rennes_model  = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="dont matter")
ollama_model  = OpenAIChatCompletionsModel(model="mistral:latest", openai_client=ollama_client)

# Gemini is optional here (Rennes is used by default below): its free tier is
# limited to 20 requests/day per model, so we don't rely on it unless you want to.
google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    gemini_model = None
    print("GOOGLE_API_KEY is not set: Gemini is optional, skipping (Rennes is used by default).")
else:
    gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
    gemini_model  = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)


instructions = f"""Here is the content of a book on the Internet of Things

{book_content}

Your responsibility is to represent the author (Laurent Toutain) for interactions with students.
The answers must be professional, clear, and should make students want to engage
in the course, or even choose this training.
If you don't know the answer to questions, respond No."""

book_agent = Agent(name="PLIDO Book Agent", instructions=instructions, model=rennes_model)

### A word on `asyncio`

You may have noticed we used `AsyncOpenAI` instead of the plain `OpenAI` client from Part 1. This is our first use of Python's `asyncio`, the standard library for writing *concurrent* code — and it matters a lot for agents.

**How it works, in short:**
* A function declared with `async def` is a **coroutine**: calling it doesn't run it immediately, it returns an object that must be driven by an **event loop**.
* Inside a coroutine, `await` hands control back to the event loop whenever we're waiting on something slow (like a network request to an LLM server), instead of blocking the whole program. The event loop can then go run other coroutines in the meantime, and come back to this one once the result is ready.
* `asyncio.gather(coro1, coro2, ...)` launches several coroutines *at the same time* and waits for all of them to finish, instead of running them one after another.
* Jupyter already runs an event loop for us, which is why we can write `await Runner.run(...)` directly in a cell, without wrapping it in `asyncio.run(...)`.

**Why it matters for agents:** most of what an agent does — calling an LLM, calling a tool, calling another agent — is *I/O-bound*: the CPU is mostly idle, waiting for a network response. In Part 1, our workflow was sequential: we waited for Ollama's answer before even starting to ask Gemini, even though the two had nothing to do with each other. With `asyncio`, we'll be able to fire off several of these calls concurrently (we'll do exactly that a bit further, asking two LLMs the same question with `asyncio.gather`), which becomes essential once an agent orchestrates many tools or sub-agents: without concurrency, response time would simply add up call after call.

An agent is launched using the Python coroutine ```run``` from the ```Runner``` module imported from OpenAI's ```agents``` module. The use of the ```await``` keyword is essential. Here, the difference with a function is minimal, since we only launch one coroutine and wait for its completion before moving to the next instruction.

You can rerun the following cell multiple times by changing the question.

In [ ]:
result = await Runner.run(book_agent, "Is this book a good book to read at the beach?")
display(Markdown(result.final_output))

That's great, we have our answer about the book, but it's a bit frustrating: we handed the keys to the LLM and have no idea what happened. A good reflex here is to ask for the request to be traced. That way, by going to the website hosting the LLM, we can see how it built its answer, as well as the protocol exchanges between our machine and the LLM server.

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    print("OPENAI_API_KEY is not set: this cell is optional (see paid_llms.md), skipping the OpenAI example.")
else:
    # the tracing exporter caches the key the first time it tries to export a trace;
    # refresh it explicitly so tracing works even if OPENAI_API_KEY was added after
    # earlier cells already ran (e.g. the Rennes/Gemini calls above) in this kernel session
    set_tracing_export_api_key(openai_api_key)

    openai_model = OpenAIChatCompletionsModel(model="gpt-5.6-luna", openai_client=AsyncOpenAI())

    openai_agent = Agent(name="OpenAI Book Agent", instructions=instructions, model=openai_model)

    with trace("Reading advice"):
        result = await Runner.run(openai_agent, "Is this book a good book to read at the beach?")
    display(Markdown(result.final_output))

If you have access to OpenAI, you got a fairly simple trace:

<img src="images/trace1.png">

It shows that all of the request's time was spent on the LLM call.

You can also see on the right that each request's instructions also include the full content of the book.

## Chat interface

We can connect the LLM queries to a graphical interface to make things friendlier. For this we'll use the `gradio` module, and at the end of the code we'll launch the chat interface. Every time the user enters a message, it will call the `chat_fn` function.

In [ ]:
async def chat_async(message, history):
    """Async function for the agent"""
    try:
        result = await Runner.run(book_agent, message)
        return result.final_output
    except Exception as e:
        return f"Error: {e}"

gradio.ChatInterface(chat_async).launch()

Here's what happens each time you type a message in the chat window: Gradio takes your text and calls `chat_async` with it, which queries the LLM agent and waits for its answer; once the answer comes back, Gradio displays it in the chat window, and the interface is ready for your next message.

`chat_async` is declared with `async def` simply because querying the agent (`await Runner.run(...)`) is itself an asynchronous operation — Gradio knows how to call this kind of function directly and wait for its result.

# Several agents

<img src="images/agents.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;"> We'll revisit a classic structure to see how to exploit parallelism with `asyncio`: we'll ask two LLMs to think about the same question and pick the better of the two answers. The goal here isn't really to get the best possible answer, it's to illustrate parallelism: both agents are queried at the same time with `asyncio.gather`, instead of waiting for one to finish before starting the other. Since one of the two will be the Ollama model running locally, chances are it won't give the smartest answer — but you never know.

In [ ]:
ollama_agent = Agent(name="PLIDO Book Agent by Ollama", instructions=instructions, model=ollama_model)

instructions= """Select the clearest answer among the different options to these student questions
about a course. Pick the one that would make you most want to take the course.
Do not add any explanation to the possible answers. Just answer with the best answer."""

best_answer  = Agent(name="Response selection", instructions=instructions, model=rennes_model)

async def chat_async(message, history):
    """Async function to call both agents, then pick the best answer"""
    
    results = await asyncio.gather( # launch both agents in parallel
        Runner.run(book_agent, message),
        Runner.run(ollama_agent, message),
    )
    outputs = [result.final_output for result in results]

    # debug: show what each model answered, before we pick the best one
    print(f"[Rennes book_agent answer]\n{outputs[0]}\n")
    print(f"[Ollama agent answer]\n{outputs[1]}\n")

    answers = "Answers to the question:\n\n" + "\nAnswer:\n".   join(outputs)
    best = await Runner.run(best_answer, answers)

    print(f"[Best answer selected]\n{best.final_output}\n")

    return best.final_output

gradio.ChatInterface(chat_async).launch()

## Amnesia

<img src="images/amnesia.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;"> You may notice that our chat doesn't remember anything. If you tell it your name, it will greet you, but if in the next question you ask it what your name is, it won't remember.

You can partially fix this by injecting into the user prompt the conversation history that gradio stores in the `history` variable, but later on we'll see much more effective techniques.

# Sending a message
<img src="images/telephone.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;">
We can ask the LLM to interact with the outside world. For this we'll use an app that sends messages to your mobile phone.

* download the `Pushover` app from your app store (Android or Apple)
* create your account
* also log in from your computer
* retrieve your user key, shown at the top right, and store it in the `PUSHOVER_USER_ID` variable in the `.env` file
* at the bottom of the page, choose *Your Applications* and click *Create an Application/Token*
  * give it a name such as *PLIDOagent*, and once validated, a token will appear
  * put this token in the `PUSHOVER_TOKEN` variable in the `.env` file

The small program below lets you test whether it works.

In [ ]:
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_user_id=os.getenv("PUSHOVER_USER_ID")
pushover_uri ="https://api.pushover.net/1/messages.json"

if not pushover_token or not pushover_user_id:
    print ("User_id or token missing")
    exit(1)

def push(message):
    payload = {"user": pushover_user_id, "token": pushover_token, "message": message}
    x = requests.post(pushover_uri, data=payload)

push("Hello my dear")


We are going to describe a function that acts as the interface between the push notification and the LLM. For this, we'll use the `@function_tool` function decorator defined by OpenAI.

In [ ]:
@function_tool
def send_message(object:str):
    """This function is used to send a message to the book author, to inform that you want to follow his class.
    If a student gives his name and wants to register uses this function to inform me.

    Args:
        - object: Protocol name.`
    """

    message = f"Student {object} is interested in the IoT course."
    push(message)
    return {"status": "success"}

We can therefore update our agent's instructions and, when creating it, add the `tools` argument, which will contain the list of programs it can call. It will use the description in the function's *docstring* to understand how to use it.

In [ ]:
prospect = False

instructions ="""
You want to recruit a student for the IoT course. Ask the student to provide their contact details, either their full name or their
email address. If you have either one, send a message to the course author using the send_message function.
"""

recruitment_agent  = Agent(name="Student recruitment", instructions=instructions, model=rennes_model,
                     tools=[send_message])


async def chat_async(message, history):
    """Async function for the agent"""
    try:
        with trace("recruitment"):
            result = await Runner.run(recruitment_agent, message)
        return result.final_output
    except Exception as e:
        return f"Error: {e}"

gradio.ChatInterface(chat_async).launch()

By using an agent on OpenAI, we can get a trace of the exchange,

<img src="images/trace2.png">

where the student provides their name:
* in the input, you can see the instructions and the user's message
* in the output, you can see the explicit call to a `send_message` tool with its arguments. The tool actually runs on your machine, but the remote LLM is the one driving it.

In a second step, the result of the `send_message` call is sent back to the agent.

Then, in a third step, a message is generated to let the user know everything went well.

# Real Agentic AI

Okay, so we can now interface functions with an LLM, so the next step is to do the same thing with Agents. In the code above, the flow is algorithmic and driven by the Python code we wrote:
* Two Agents are called to provide answers
* a third Agent analyzes the answers, picks the best one, and if it detects a reference to a protocol, sends an alert to the professor.

We're now going to change the logic of the code by turning the two Agents responsible for the answers into functions, and having the third Agent orchestrate the whole process — that is, call the functions (former Agents) for the answers, and send a message if it detects a protocol.

In [ ]:
tool_answer1 = book_agent.as_tool(tool_name="Book_agent", tool_description="answer to user questions")
tool_answer2 = recruitment_agent.as_tool(tool_name="Recruitment_agent", tool_description="Ask the student's name")

tools = [tool_answer1, tool_answer2]

instructions ="""
You handle registrations for the IoT course. Students will ask you questions about the course content, 
which is also covered in the book. The book_agent can answer these technical questions. If 
the student wants to register, use the Recruitment_agent to send it to the professor via the send_message function.
"""

# gpt-5.6-luna is a reasoning model: it rejects function tools on /v1/chat/completions
# unless reasoning effort is explicitly turned off
global_agent = Agent("Global Agent", instructions=instructions, tools=tools, model=openai_model,
                      model_settings=ModelSettings(reasoning={"effort": "none"}))

async def chat_async(message, history):
    """Async function to orchestrate the calls to the tools/agents"""
    
    with trace("agentic"):
        result = await Runner.run(global_agent, message)

    return result.final_output

gradio.ChatInterface(chat_async).launch()